#  Goal 2 — Implementação e carregamento em MySQL

Neste notebook criamos as 6 tabelas de SQL (`movies`, `genres`, `movie_genres`, `users`, `ratings` e `tags`), a partir dos CSVs limpos em `data/processed/`. Medimos o tempo de carregamento e verificamos se os dados ficaram completos.

Importar as bibliotecas e ligar ao MySQL

In [4]:
import pandas as pd
import mysql.connector
import time
from pathlib import Path 

conn = mysql.connector.connect(
    host="localhost",
    port=3306,
    user="root",
    password="LLX031219"  #cada um depois alterar para a sua password
)

cursor = conn.cursor()

print("Ligação ao MySQL estabelecida com sucesso!")

Ligação ao MySQL estabelecida com sucesso!


## 1. Ler os dados

Lemos os seis ficheiros CSV previamente preparados em `data/processed/`. Convertemos os valores em falta para `None`, que serão representados como `NULL` no MySQL, e os campos de data para o formato datetime.

In [5]:
ficheiros = [
    ("movies", {"movieId": "Int64", "title": "str", "year": "Int64", "imdbId": "str", "tmdbId": "Int64"}, None),
    ("genres", {"genre": "str", "genreId": "Int64"}, None),
    ("movie_genres", {"movieId": "Int64", "genreId": "Int64"}, None),
    ("users", {"userId": "Int64", "country": "str"}, None),
    ("ratings", {"userId": "Int64", "movieId": "Int64", "rating": "Float64"}, ["timestamp"]),
    ("tags", {"userId": "Int64", "movieId": "Int64", "tag": "str"}, ["timestamp"])
]
dados = {}

for nome, tipos, data in ficheiros:
    df = pd.read_csv(f"../data/processed/{nome}.csv", dtype=tipos, parse_dates = data)
    dados[nome] = df.astype(object).where(df.notna(), None).to_dict("records")
    print(nome, len(dados[nome]))

print(dados["ratings"][0])

movies 9742
genres 19
movie_genres 22050
users 610
ratings 100000
tags 3683
{'userId': 432, 'movieId': 77866, 'rating': 4.5, 'timestamp': Timestamp('2012-04-23 00:07:21')}


In [6]:
print(dados["users"][0])

{'userId': 1, 'latitude': 40.71862834068687, 'longitude': -7.827198594667862}


## 2. Criar as tabelas

Criamos as **6 tabelas** no MySQL (`movies`, `genres`, `movie_genres`, `users`, `ratings` e `tags`) através de `CREATE TABLE`, definindo os tipos de dados e as restrições, como `NOT NULL`, `PRIMARY KEY`, `FOREIGN KEY` e `CHECK` (por exemplo, o `rating` tem de estar entre 0.5 e 5).

As **chaves primárias (PK)** garantem que não existem registos com a mesma chave. Algumas tabelas utilizam **chaves primárias compostas**, como `ratings`, que combina `userId` e `movieId`. As **chaves estrangeiras (FK)** garantem as relações entre as tabelas.

Estas restrições permitem manter a **integridade dos dados** e estabelecer regras equivalentes às do MongoDB, para comparar os tempos de carregamento das duas bases de dados.


In [7]:
# criar a base de dados se não existir
cursor.execute("CREATE DATABASE IF NOT EXISTS bda_movielens")

# escolher a base de dados
cursor.execute("USE bda_movielens")

print("Base de dados selecionada com sucesso!")

Base de dados selecionada com sucesso!


In [8]:

# limpar as tabelas antigas
for tabela in [
    "ratings",
    "tags",
    "movie_genres",
    "users",
    "movies",
    "genres"
]:
    cursor.execute(f"DROP TABLE IF EXISTS `{tabela}`")

print("Tabelas antigas removidas com sucesso!")


Tabelas antigas removidas com sucesso!


In [ ]:
cursor.execute("""
CREATE TABLE IF NOT EXISTS users (
    userId INT PRIMARY KEY,
    latitude DOUBLE NOT NULL,
    longitude DOUBLE NOT NULL
)
""")

cursor.execute("""
CREATE TABLE IF NOT EXISTS movies (
    movieId INT PRIMARY KEY,
    title VARCHAR(512) NOT NULL,
    year INT NULL,
    imdbId VARCHAR(7) NULL,
    tmdbId INT NULL
)
""")

cursor.execute("""
CREATE TABLE IF NOT EXISTS genres (
    genreId INT PRIMARY KEY,
    genre VARCHAR(100) NOT NULL UNIQUE
)
""")


cursor.execute("""
CREATE TABLE IF NOT EXISTS ratings (
    userId INT NOT NULL,
    movieId INT NOT NULL,
    rating DECIMAL(2,1) NOT NULL,
    timestamp DATETIME NOT NULL,

    PRIMARY KEY (userId, movieId),

    FOREIGN KEY (userId) REFERENCES users(userId),
    FOREIGN KEY (movieId) REFERENCES movies(movieId),

    CONSTRAINT chk_rating CHECK (rating BETWEEN 0.5 AND 5.0)
)
""")

cursor.execute("""
CREATE TABLE IF NOT EXISTS tags (
    tagId INT AUTO_INCREMENT PRIMARY KEY,
    userId INT NOT NULL,
    movieId INT NOT NULL,
    tag TEXT NOT NULL,
    timestamp DATETIME NOT NULL,

    FOREIGN KEY (userId) REFERENCES users(userId),
    FOREIGN KEY (movieId) REFERENCES movies(movieId)
)
""")

cursor.execute("""
CREATE TABLE IF NOT EXISTS movie_genres (
    movieId INT NOT NULL,
    genreId INT NOT NULL,

    PRIMARY KEY (movieId, genreId),

    FOREIGN KEY (movieId) REFERENCES movies(movieId),
    FOREIGN KEY (genreId) REFERENCES genres(genreId)
)
""")

print("As 6 tabelas foram criadas com sucesso!")


As 6 tabelas foram criadas com sucesso!


In [10]:
# ver as tabelas criadas
cursor.execute("SHOW TABLES")
for tabela in cursor.fetchall():
    print(tabela[0])

genres
movie_genres
movies
ratings
tags
users



## 3. Carregar os dados e medir o tempo

Inserimos os registos nas **6 tabelas** do MySQL através de `INSERT INTO` e `executemany()`, utilizando os dados dos ficheiros CSV previamente preparados.

Medimos o **tempo de carregamento** de cada tabela e o tempo total com `time.perf_counter()`, incluindo a confirmação das transações com `commit()`, para comparar os resultados com o MongoDB.


In [11]:
tempos_mysql = {}
ordem = [
    "users",
    "movies",
    "genres",
    "ratings",
    "tags",
    "movie_genres"
]

for nome in ordem:
    registos = dados[nome]

    if registos:
        colunas = ", ".join(registos[0].keys())
        valores = ", ".join(["%s"] * len(registos[0]))

        sql = f"INSERT INTO {nome} ({colunas}) VALUES ({valores})"

        inicio = time.perf_counter()

        cursor.executemany(
            sql,
            [tuple(r.values()) for r in registos]
        )

        conn.commit()
        tempos_mysql[nome] = time.perf_counter() - inicio
        print(
            f"{nome}: {len(registos)} registos "
            f"em {tempos_mysql[nome]:.2f} s"
        )

print(f"Total MySQL: {sum(tempos_mysql.values()):.2f} s")


ProgrammingError: 1054 (42S22): Unknown column 'latitude' in 'field list'

## 4. Verificar os dados

Confirmamos que cada **tabela** do MySQL contém o mesmo número de **registos** que o respetivo ficheiro CSV, utilizando `SELECT COUNT(*)`.

Também mostramos os **índices** existentes em cada tabela através de `SHOW INDEX`, incluindo os índices associados às **chaves primárias (PK)** e às **chaves estrangeiras (FK)**, para verificar a estrutura da base de dados.


In [ ]:

# Verificar o número de registos
for nome, registos in dados.items():
    cursor.execute(f"SELECT COUNT(*) FROM `{nome}`")
    total = cursor.fetchone()[0]
    print(f"{nome}: CSV = {len(registos)}, MySQL = {total}")
    assert total == len(registos), f"Número de registos diferente em {nome}"

print()

# Mostrar os índices de cada tabela
for nome in dados:
    cursor.execute(f"SHOW INDEX FROM `{nome}`")
    indices = sorted(set(linha[2] for linha in cursor.fetchall()))
    print(f"Índices de {nome}: {indices}")


movies: CSV = 9742, MySQL = 9742
genres: CSV = 19, MySQL = 19
movie_genres: CSV = 22050, MySQL = 22050
users: CSV = 610, MySQL = 610
ratings: CSV = 100000, MySQL = 100000
tags: CSV = 3683, MySQL = 3683

Índices de movies: ['PRIMARY']
Índices de genres: ['PRIMARY', 'genre']
Índices de movie_genres: ['PRIMARY', 'genreId']
Índices de users: ['PRIMARY']
Índices de ratings: ['PRIMARY', 'movieId']
Índices de tags: ['PRIMARY', 'movieId', 'userId']
